In [ ]:
#=====CELL 1: IMPORTS & SETUP======
# Import all necessary libraries
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import numpy as np
import warnings
from pandas.errors import SettingWithCopyWarning

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=SettingWithCopyWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print("Libraries imported successfully.")

: 

In [ ]:
#=====CELL 2: DEFINE CONSTANTS & BOUNDING BOX======
# Define the approximate bounding box for Washington County, Utah
# We expanded the MAXY to 37.7 to ensure we catch the northern county data
MINX, MINY, MAXX, MAXY = (-114.1, 37.0, -112.9, 37.7)

# Define a standard Coordinate Reference System (CRS) for the project.
# EPSG:4326 is WGS84 (standard latitude/longitude)
PROJECT_CRS = "EPSG:4326"

print(f"Project CRS set to: {PROJECT_CRS}")
print(f"Washington County Bounding Box defined: {MINX, MINY, MAXX, MAXY}")

In [ ]:
#=====CELL 3: LOAD & PRE-FILTER FIRE DATA======
# Load the raw fire data
df_fire_raw = gpd.read_file('../../data/raw/fire/USA-Fire-Area.geojson')

# Re-project to our standard CRS immediately so spatial filtering works
df_fire_raw = df_fire_raw.to_crs(PROJECT_CRS)

# --- Task 5: Filtering (Record-Based) ---
# Filter 1: Keep only rows where the state column is 'UT'
fires_utah = df_fire_raw[df_fire_raw['state'].str.upper() == 'UT'].copy()

# Filter 2: Use a robust polygon intersection to filter for Washington County
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
fires_washington = fires_utah[fires_utah.geometry.intersects(washington_bbox_poly)].copy()

print(f"Original fire records: {len(df_fire_raw)}")
print(f"Utah fire records: {len(fires_utah)}")
print(f"Washington Co. fire records: {len(fires_washington)}")

# Sanity Check
if len(fires_washington) == 0:
    print("\n*** WARNING: No fire records found. Check Bounding Box. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(fires_washington)} fire records. ***\n")

In [ ]:
#=====CELL 4: CLEAN & TRANSFORM FIRE DATA======

# --- Task 5: Filtering (Field-Based) ---
keep_cols_fire = [
    'incidentname', 'fireyear', 'gisacres', 
    'perimeterdatetime', 'agency', 'geometry'
]
fires_clean = fires_washington[keep_cols_fire].copy()

# Rename columns
fires_clean.rename(columns={
    'incidentname': 'name',
    'fireyear': 'year',
    'gisacres': 'acres_burned',
    'perimeterdatetime': 'event_datetime',
    'agency': 'managing_agency'
}, inplace=True)

# --- Part 3: Attribute Data Cleaning ---
fires_clean['event_datetime'] = pd.to_datetime(fires_clean['event_datetime'], errors='coerce')
fires_clean.dropna(subset=['event_datetime', 'geometry'], inplace=True)

# --- Task 1 & 2: Extraction & Derivation ---
fires_clean['event_hour'] = fires_clean['event_datetime'].dt.hour
fires_clean['event_month'] = fires_clean['event_datetime'].dt.month
fires_clean['year'] = fires_clean['year'].astype(int)

# --- Task 4: Binning ---
bins_fire = [0, 100, 1000, 10000, np.inf]
labels_fire = ['Small', 'Medium', 'Large', 'Very Large']
fires_clean['damage_level'] = pd.cut(fires_clean['acres_burned'], bins=bins_fire, labels=labels_fire, right=False)
fires_clean['damage_level'] = fires_clean['damage_level'].cat.add_categories('Minimal').fillna('Minimal')

# --- Part 3: Spatial Cleaning ---
# Fix invalid geometries
invalid_geoms = ~fires_clean.geometry.is_valid
if invalid_geoms.any():
    print(f"Fixing {invalid_geoms.sum()} invalid fire geometries...")
    fires_clean.loc[invalid_geoms, 'geometry'] = fires_clean.loc[invalid_geoms, 'geometry'].buffer(0)

fires_clean['hazard_type'] = 'wildfire'
fires_clean.drop_duplicates(subset=['event_datetime', 'geometry'], inplace=True)

print(f"Fire data cleaned. {len(fires_clean)} records remaining.")

In [ ]:
#=====CELL 5: LOAD & PRE-FILTER EARTHQUAKE DATA======
# Load the raw earthquake data
df_earthquake_raw = pd.read_csv('../../data/raw/earth_quakes/usgs_main.csv')

# Convert to GeoDataFrame
earthquake_gdf = gpd.GeoDataFrame(
    df_earthquake_raw,
    geometry=gpd.points_from_xy(df_earthquake_raw['longitude'], df_earthquake_raw['latitude']),
    crs=PROJECT_CRS
)

# --- Task 5: Filtering ---
# Use robust polygon check
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
earthquake_washington = earthquake_gdf[earthquake_gdf.geometry.within(washington_bbox_poly)].copy()

print(f"Original earthquake records: {len(df_earthquake_raw)}")
print(f"Washington Co. earthquake records: {len(earthquake_washington)}")

# Sanity Check
if len(earthquake_washington) == 0:
    print("\n*** WARNING: No earthquake records found. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(earthquake_washington)} earthquake records. ***\n")

In [ ]:
#=====CELL 6: CLEAN & TRANSFORM EARTHQUAKE DATA======

# --- Task 5: Filtering (Field-Based) ---
keep_cols_eq = [
    'time', 'latitude', 'longitude', 'depth', 
    'mag', 'place', 'geometry'
]
earthquake_clean = earthquake_washington[keep_cols_eq].copy()

# Rename columns
earthquake_clean.rename(columns={
    'time': 'event_datetime',
    'mag': 'magnitude',
    'depth': 'depth_km',
    'place': 'location_desc'
}, inplace=True)

# --- Part 3: Attribute Cleaning ---
earthquake_clean['event_datetime'] = pd.to_datetime(earthquake_clean['event_datetime'], errors='coerce')
earthquake_clean.dropna(subset=['event_datetime', 'magnitude', 'geometry'], inplace=True)

# --- Task 1: Derivation ---
earthquake_clean['event_month'] = earthquake_clean['event_datetime'].dt.month
earthquake_clean['event_year'] = earthquake_clean['event_datetime'].dt.year

# --- Task 4: Binning ---
bins_eq = [-np.inf, 2.5, 5.4, 6.0, 6.9, np.inf]
labels_eq = ['Minimal', 'Light', 'Moderate', 'Strong', 'Major']
earthquake_clean['danger_level'] = pd.cut(earthquake_clean['magnitude'], bins=bins_eq, labels=labels_eq, right=False)

earthquake_clean['hazard_type'] = 'earthquake'
earthquake_clean.drop_duplicates(subset=['event_datetime', 'magnitude', 'location_desc'], inplace=True)

print(f"Earthquake data cleaned. {len(earthquake_clean)} records remaining.")

In [ ]:
#=====CELL 7: LOAD & PRE-FILTER FLOOD DATA======
# Load the raw flood data
df_flood_raw = gpd.read_file('../../data/raw//flood_data/flood_hazard.geojson')

# Reproject before filtering
df_flood_raw = df_flood_raw.to_crs(PROJECT_CRS)

# --- Task 5: Filtering ---
washington_bbox_poly = box(MINX, MINY, MAXX, MAXY)
flood_washington = df_flood_raw[df_flood_raw.geometry.intersects(washington_bbox_poly)].copy()
# Clip geometries to keep it clean
flood_washington['geometry'] = flood_washington['geometry'].intersection(washington_bbox_poly)

print(f"Original flood zone records: {len(df_flood_raw)}")
print(f"Washington Co. flood zone records: {len(flood_washington)}")

# Sanity Check
if len(flood_washington) == 0:
    print("\n*** WARNING: No flood zones found. ***\n")
else:
    print(f"\n*** SUCCESS: Found {len(flood_washington)} flood zones. ***\n")

In [ ]:
#=====CELL 8: CLEAN & TRANSFORM FLOOD DATA======

# --- Task 5: Filtering ---
keep_cols_flood = ['fld_zone', 'zone_subty', 'depth', 'geometry']
flood_clean = flood_washington[keep_cols_flood].copy()

# --- Part 3: Attribute Cleaning ---
flood_clean['fld_zone'] = flood_clean['fld_zone'].str.upper().str.strip()
flood_clean['zone_subty'] = flood_clean['zone_subty'].fillna('None')

# --- Task 4: Binning ---
bins_flood = [-np.inf, 0, 1, 3, 5, np.inf]
labels_flood = ['Unknown', 'Minimal (0ft)', 'Low (1-3ft)', 'Medium (3-5ft)', 'High (5+ft)']
flood_clean['danger_level'] = pd.cut(flood_clean['depth'], bins=bins_flood, labels=labels_flood, right=True)

# --- Part 3: Spatial Cleaning ---
flood_clean.dropna(subset=['geometry'], inplace=True)
flood_clean = flood_clean[~flood_clean.geometry.is_empty]

invalid_geoms = ~flood_clean.geometry.is_valid
if invalid_geoms.any():
    print(f"Fixing {invalid_geoms.sum()} invalid flood geometries...")
    flood_clean.loc[invalid_geoms, 'geometry'] = flood_clean.loc[invalid_geoms, 'geometry'].buffer(0)

flood_clean['hazard_type'] = 'flood_zone'
flood_clean.drop_duplicates(subset=['fld_zone', 'geometry'], inplace=True)

print(f"Flood data cleaned. {len(flood_clean)} records remaining.")

In [ ]:
#=====CELL 9: SAVE CLEANED INDIVIDUAL DATASETS======
# Save the three clean datasets for reference
fires_clean.to_file('../../data/processed/washington_fires_clean.geojson', driver='GeoJSON')
earthquake_clean.to_file('../../data/processed/washington_earthquakes_clean.geojson', driver='GeoJSON')
flood_clean.to_file('../../data/processed/washington_floods_clean.geojson', driver='GeoJSON')
print("Saved all intermediate cleaned files.")

In [ ]:
#=====CELL 10: CREATE LOCATION-BASED HAZARD DATASET (SPATIAL JOIN)======
# --- Task 3, Task 6, & Part 2 (Data Blending) ---

# Define a Projected CRS (UTM Zone 12N for Utah meters) and Buffer Size (5km)
UTM_CRS = "EPSG:26912" 
BUFFER_METERS = 5000
print(f"Using projected CRS: {UTM_CRS} with a {BUFFER_METERS}m buffer.")

# Step 1: Create a grid of points (0.01 deg ~ 1.1km)
x = np.arange(MINX, MAXX, 0.01)
y = np.arange(MINY, MAXY, 0.01)
x_coords, y_coords = np.meshgrid(x, y)
points = [Point(x, y) for x, y in zip(x_coords.flatten(), y_coords.flatten())]
danger_grid = gpd.GeoDataFrame(geometry=points, crs=PROJECT_CRS)
print(f"Created a danger grid with {len(danger_grid)} points.")

# Project all datasets to UTM for accurate joining
print("Projecting all datasets to UTM CRS...")
danger_grid_proj = danger_grid.to_crs(UTM_CRS)
fires_proj = fires_clean.to_crs(UTM_CRS)
earthquakes_proj = earthquake_clean.to_crs(UTM_CRS)
flood_proj = flood_clean.to_crs(UTM_CRS)

# Step 2: Join with Flood Data (Projected)
grid_with_flood = gpd.sjoin(danger_grid_proj, flood_proj, how='left', predicate='within')
grid_with_flood.rename(columns={'fld_zone': 'flood_zone', 'danger_level': 'flood_danger'}, inplace=True)
grid_with_flood = grid_with_flood[['flood_zone', 'flood_danger']]
print("Spatially joined grid with flood zones.")

# Step 3: Buffer Grid and Join with Events
danger_grid_buffered_proj = danger_grid_proj.buffer(BUFFER_METERS)
print(f"Buffered grid points by {BUFFER_METERS} meters.")

# Join Fires
fires_sjoined = gpd.sjoin(
    gpd.GeoDataFrame(geometry=danger_grid_buffered_proj, crs=UTM_CRS), 
    fires_proj, how='left', predicate='intersects'
)
fire_counts = fires_sjoined.groupby(fires_sjoined.index)['index_right'].count()
fire_avg_acres = fires_sjoined.groupby(fires_sjoined.index)['acres_burned'].mean()
print("Aggregated fire counts per grid cell.")

# Join Earthquakes
earthquakes_sjoined = gpd.sjoin(
    gpd.GeoDataFrame(geometry=danger_grid_buffered_proj, crs=UTM_CRS),
    earthquakes_proj, how='left', predicate='intersects'
)
eq_counts = earthquakes_sjoined.groupby(earthquakes_sjoined.index)['index_right'].count()
eq_max_mag = earthquakes_sjoined.groupby(earthquakes_sjoined.index)['magnitude'].max()
print("Aggregated earthquake counts per grid cell.")

# Step 4: Combine back to original Lat/Lon Grid
danger_grid['fire_count'] = fire_counts
danger_grid['fire_avg_acres'] = fire_avg_acres
danger_grid['earthquake_count'] = eq_counts
danger_grid['earthquake_max_mag'] = eq_max_mag

final_hazards_by_location = danger_grid.merge(
    grid_with_flood, left_index=True, right_index=True, how='left'
)

# Fill NaNs
final_hazards_by_location['fire_count'].fillna(0, inplace=True)
final_hazards_by_location['fire_avg_acres'].fillna(0, inplace=True)
final_hazards_by_location['earthquake_count'].fillna(0, inplace=True)
final_hazards_by_location['earthquake_max_mag'].fillna(0, inplace=True)

# Handle Categorical Filling safely
if not isinstance(final_hazards_by_location['flood_danger'].dtype, pd.CategoricalDtype):
     final_hazards_by_location['flood_danger'] = pd.Categorical(
         final_hazards_by_location['flood_danger'], 
         categories=flood_clean['danger_level'].cat.categories
     )

final_hazards_by_location['flood_zone'].fillna('None', inplace=True)
final_hazards_by_location['flood_danger'] = final_hazards_by_location['flood_danger'].cat.add_categories('None')
final_hazards_by_location['flood_danger'].fillna('None', inplace=True)

print("\nFinal location-based hazard dataset created.")

In [ ]:
#=====CELL 11: SAVE FINAL COMBINED DATASET======
final_hazards_by_location.to_file(
    '../../data/processed/hazards_by_location_washington.geojson', 
    driver='GeoJSON'
)
print(f"Successfully saved final dataset with {len(final_hazards_by_location)} grid points.")

In [ ]:
#=====CELL 12: VERIFY JOIN RESULTS (SANITY CHECK)======
print("--- Sanity Check for Hazard Joins ---")
print(f"Total fire count (sum): {final_hazards_by_location['fire_count'].sum()}")
print(f"Highest 'fire_count' in one cell: {final_hazards_by_location['fire_count'].max()}")
print(f"Total earthquake count (sum): {final_hazards_by_location['earthquake_count'].sum()}")
print(f"Max magnitude found: {final_hazards_by_location['earthquake_max_mag'].max()}")
print("Flood zone distribution:")
print(final_hazards_by_location['flood_zone'].value_counts().head())

In [ ]:
#=====CELL 13: CREATE FINAL 'DANGER_LEVEL'======
# Revised scoring based on your actual data ranges
print("\n--- Creating Final Danger Level ---")

final_hazards_by_location['danger_score'] = 0

# Fire Scoring
final_hazards_by_location.loc[final_hazards_by_location['fire_count'] > 0, 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['fire_count'] > 5, 'danger_score'] += 1

# Earthquake Scoring (Threshold adjusted to 2.0 to catch actual data)
final_hazards_by_location.loc[final_hazards_by_location['earthquake_count'] > 0, 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['earthquake_max_mag'] > 2.0, 'danger_score'] += 1

# Flood Scoring
final_hazards_by_location.loc[final_hazards_by_location['flood_zone'] != 'None', 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['flood_zone'].isin(['A', 'AE', 'AO']), 'danger_score'] += 1
final_hazards_by_location.loc[final_hazards_by_location['flood_danger'].isin(['Medium (3-5ft)', 'High (5+ft)']), 'danger_score'] += 1

# Binning the final score
bins = [-1, 0, 1, 2, 3, np.inf] 
labels = ['Low', 'Moderate', 'High', 'Very High', 'Extreme']
final_hazards_by_location['danger_level'] = pd.cut(
    final_hazards_by_location['danger_score'], 
    bins=bins, 
    labels=labels
)

print("Final Danger Level Distribution:")
print(final_hazards_by_location['danger_level'].value_counts())

print("\n--- Top 5 Most Dangerous Locations ---")
print(final_hazards_by_location.sort_values(by='danger_score', ascending=False).head())